# 📝 텍스트 전처리·분석 과제 LV3 정답 — 텍스트 리포트·연관어 탐색 (강사용)

각 단계의 **모범 코드 + 자가채점 + 해설** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day11_텍스트전처리_분석/data/`·`output/`·`../../day11_텍스트전처리_분석/images/` 입니다.
- 워드클라우드는 반드시 `font_path=FONT_PATH` 를 넘깁니다(안 넣으면 한글이 전부 □□□).
- TF-IDF 점수는 실수라 라이브러리 버전에 따라 소수점이 달라질 수 있어 **순위·포함 여부·개수**로만 채점합니다. 빈도·bigram·동시 출현은 정수라 값까지 확인합니다.

아래 셀을 먼저 실행해 이 단원에 필요한 라이브러리와 한글 폰트·형태소 분석기를 준비하세요. (실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

## 1. 리뷰 텍스트 분석 리포트 생성기
**배경**: 한 선크림 상품에 달린 소비자 리뷰 534건을 **정제 → 토큰화 → 불용어 정리 → 빈도·핵심어 추출** 까지 한 흐름으로 처리해, **만족 고객과 불만 고객이 각각 어떤 말을 하는지** 를 담은 리포트 표를 만들고 CSV 로 저장하는 미니 프로그램을 완성합니다.

아래 각 `### N단계` 셀의 지시대로 이어서 풉니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | `df` 534행, 일반 불용어 `stopwords` 679개 |
| 2단계 | 전처리 함수 `clean_text`·`tokenize` 완성 — 예시 문자열로 결과 확인 |
| 3단계 | **도메인 불용어** `domain_stopwords` 를 만들어 재적용 — 전후 top10 비교 |
| 4단계 | 빈도 top15 막대그래프 + 워드클라우드(완성 그래프처럼) |
| 5단계 | TF-IDF(`min_df=5`) → **차이 기반 특징어** `top_satisfied`·`top_dissatisfied` 각 10개 |
| 6단계 | **별점별(1~5) 상위 키워드** `rating_keywords` + 별점별 키워드 히트맵(완성 그래프처럼) |
| 7단계 | 리포트 `report` 컬럼 `{group, rank, word}`, 20행 → `output/text_report.csv` 저장 |
| 인사이트 | 만족·불만 특징어와 별점별 변화로 읽는 소비자 관심사 2~3문장 서술 |

### 1단계 — 데이터·불용어 로드와 살펴보기
`../../day11_텍스트전처리_분석/data/reviews_sun.csv` 를 `df` 로 불러오고, 일반 한국어 불용어 목록 `../../day11_텍스트전처리_분석/data/stopwords_ko.json`(JSON 배열)을 읽어 **집합(set)** `stopwords` 변수에 담으세요. 그리고 리뷰 수·불용어 수를 출력하고, `df.head()`·`df.info()`·수치형/범주형 요약 통계와 별점 분포(`rating` 값별 개수)를 살펴보세요.

- **요구 변수**: `df`(534행, 컬럼 `rating`·`text`), `stopwords`(불용어 **집합**, 679개).
- **주의**: 불용어는 뒤 단계에서 `단어 not in stopwords` 로 계속 조회하니 리스트가 아니라 **집합**으로 만드세요(조회가 훨씬 빠릅니다). JSON 은 `json.load` 로 읽습니다.

<details><summary>힌트</summary>

```text
접근방법:
- CSV 를 데이터프레임으로 읽고, JSON 불용어 배열을 읽어 집합으로 바꾼다.

세부구현:
1. read_csv 로 df 를 만든다
2. open 으로 JSON 파일을 열어 json.load 로 리스트를 읽고 set 으로 바꿔 `stopwords` 변수에 담는다
3. 리뷰 수와 불용어 수를 출력한다
4. df.head(), info(), 수치형·범주형 describe(), rating 값별 개수를 살펴본다
```

</details>

In [ ]:
df = pd.read_csv("../../day11_텍스트전처리_분석/data/reviews_sun.csv")
with open("../../day11_텍스트전처리_분석/data/stopwords_ko.json", encoding="utf-8") as f:
    stopwords = set(json.load(f))
print("리뷰 수:", len(df), "일반 불용어 수:", len(stopwords))
display(df.head())
df.info()
display(df.describe(include="number"))
display(df.describe(exclude="number"))
print("별점 분포:")
display(df["rating"].value_counts().sort_index())

In [ ]:
# [자가채점]
assert len(df) == 534
assert set(df.columns) == {"rating", "text"}
assert isinstance(stopwords, set) and len(stopwords) == 679
print("✅ 1단계 통과!")

### 해설 — 문제 1 · 1단계 — 데이터·불용어 로드
- **접근법**: CSV 와 불용어 JSON 을 먼저 읽고 `head`·`shape`·별점 분포를 확인합니다. 텍스트 분석도 **데이터를 먼저 훑는 것**에서 시작합니다.
- **흔한 실수**: 불용어 파일을 리스트 그대로 두는 것입니다. 뒤에서 `word not in stopwords` 를 수만 번 호출하므로 **집합(set)으로 바꿔야** 속도가 크게 빨라집니다.
- **대안**: 별점 분포가 한쪽으로 치우치면 표본이 적은 그룹의 결과는 **불확실성이 더 큽니다**. 그룹별 건수를 함께 표시하고, 결과를 단정하지 말고 경향으로 해석하세요.

### 2단계 — 전처리 파이프라인 완성 (clean_text · tokenize)
리뷰 원문에는 특수문자·이모지·`ㅋㅋㅋㅋ` 같은 반복 문자가 섞여 있습니다. 이 단원에서 배운 **정규표현식 정제 + 형태소 토큰화** 를 두 함수로 만드세요.

**(1) `clean_text(text)`** — `re.sub` 세 번으로 아래 순서대로 처리하고 앞뒤 공백을 없앤 문자열을 반환합니다.

| 순서 | 하는 일 | 규칙 |
| --- | --- | --- |
| 1 | 특수문자·이모지 제거 | 한글·영문·숫자·공백이 **아닌** 문자를 **공백 한 칸**으로 바꾼다 |
| 2 | 반복 문자 축약 | 같은 문자가 **3번 이상** 이어지면 **2번**으로 줄인다 |
| 3 | 공백 정규화 | 연속 공백을 한 칸으로 줄이고 앞뒤 공백을 없앤다 |

**(2) `tokenize(text, stopwords)`** — 정제한 문장을 `kiwi.tokenize(...)` 로 형태소 분석해, 아래 세 조건을 **모두** 만족하는 형태소의 표면형(`.form`)만 **리스트**로 반환합니다.

- 품사 태그(`.tag`)가 `NN`(명사)·`VA`(형용사)·`VV`(동사) 중 하나로 **시작**
- 표면형 길이가 **2글자 이상**
- 불용어 집합에 **없음**

- **요구 함수**: `clean_text(text)` → 문자열, `tokenize(text, stopwords)` → 문자열 리스트.
- **예시** (아래 값이 그대로 나와야 합니다)

| 입력 | `clean_text` 결과 | `tokenize` 결과 |
| --- | --- | --- |
| `'정말 좋아요오오오!!! 끈적임 없고 촉촉해요~~ ^^'` | `'정말 좋아요오오 끈적임 없고 촉촉해요'` | `['끈적이', '촉촉하']` |

- **주의**: `str(text)` 로 감싸 결측값도 안전하게 처리하세요. `kiwi.tokenize` 는 형태소 객체 리스트를 돌려주며 각 객체의 `.form`(표면형)·`.tag`(품사)를 씁니다. 여러 태그로 시작하는지 한 번에 보려면 `str.startswith` 에 **튜플**을 넘길 수 있어요.

<details><summary>힌트</summary>

```text
접근방법:
- 정제는 정규표현식 치환 세 번으로, 토큰화는 형태소 분석 결과를 조건으로 걸러 만든다.

세부구현:
1. clean_text 를 정의한다
   1-1. 한글·영문·숫자·공백이 아닌 문자를 공백으로 치환한다
   1-2. 같은 문자가 3번 이상 반복되면 2번으로 줄인다 (뒤따라오는 반복을 잡는 패턴 + 역참조)
   1-3. 연속 공백을 한 칸으로 줄이고 strip 으로 앞뒤 공백을 없앤 뒤 반환한다
2. tokenize 를 정의한다
   2-1. clean_text 로 먼저 정제한 문장을 kiwi 로 형태소 분석한다
   2-2. 태그가 명사·형용사·동사로 시작하고, 길이가 1보다 크고, 불용어에 없는 형태소만 남긴다
   2-3. 남은 형태소의 표면형만 리스트로 반환한다
3. 예시 문자열로 두 함수를 실행해 결과를 눈으로 확인한다
```

</details>

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', text)   # 특수문자·이모지 제거
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)              # 반복 문자 축약
    text = re.sub(r'\s+', ' ', text)                        # 공백 정규화
    return text.strip()

def tokenize(text, stopwords):
    return [t.form for t in kiwi.tokenize(clean_text(text))
            if t.tag.startswith(('NN', 'VA', 'VV'))          # 명사·형용사·동사만
            and len(t.form) > 1 and t.form not in stopwords]

sample = "정말 좋아요오오오!!! 끈적임 없고 촉촉해요~~ ^^"
print("정제:", clean_text(sample))
print("토큰:", tokenize(sample, stopwords))

In [ ]:
# [자가채점]
sample = "정말 좋아요오오오!!! 끈적임 없고 촉촉해요~~ ^^"
assert clean_text(sample) == "정말 좋아요오오 끈적임 없고 촉촉해요"
assert tokenize(sample, stopwords) == ["끈적이", "촉촉하"]
print("✅ 2단계 통과!")

### 해설 — 문제 1 · 2단계 — 전처리 파이프라인
- **접근법**: `clean_text`(정규식 정제)와 `tokenize`(형태소 + 품사 필터 + 불용어)를 **함수 두 개로 나눠** 만듭니다. 나눠 두면 정제 규칙만 바꿔 실험하기 쉽습니다.
- **흔한 실수**: 형태소 분석기를 함수 안에서 매번 새로 만드는 것입니다. `Kiwi()` 생성은 무겁기 때문에 **바깥에서 한 번만 만들어** 재사용해야 합니다.
- **대안**: 품사를 명사만(NNG·NNP)으로 좁히면 주제어가 선명해지고, 형용사·동사까지 넣으면 감정 표현이 살아납니다. 분석 목적에 따라 고르세요.

### 3단계 — 도메인 불용어 구축과 재적용
일반 불용어만으로는 부족합니다. **빈도 상위를 직접 눈으로 보고, 이 도메인에서만 의미가 없는 고빈도어를 골라 목록을 만드는 것** 이 실무의 핵심입니다.

1. 리뷰 534건을 모두 `tokenize(text, stopwords)` 로 토큰화해 **리뷰별 토큰 리스트**를 `tokens_general` 변수에 담고(길이 534), 전체 단어 빈도를 `Counter` 로 세어 **top20 을 출력**하세요.
2. 그 top20 을 보면 아래처럼 **어느 상품 리뷰에나 나오는 리뷰 메타어**가 섞여 있습니다. 이런 단어를 모아 **집합** `domain_stopwords` 를 만드세요 — 아래 **14개**입니다.

`제품·사용·구매·상품·주문·배송·리뷰·느낌·생각·쿠팡·체험·무료·제공·이벤트`

```text
제품  사용  구매  상품  주문  배송  리뷰  느낌  생각
```

3. `domain_stopwords` 를 **일반 불용어와 합쳐** 다시 토큰화해 `tokens` 변수에 담고(길이 534), 빈도를 다시 세어 **top10 을 비교**하세요.
4. 비교용 변수: 3단계 적용 **전** top10 단어 리스트 `top10_before`, **후** top10 단어 리스트 `top10_after`(둘 다 단어 문자열 10개짜리 리스트, 빈도 내림차순).

- **요구 변수**: `domain_stopwords`(집합, 14개), `tokens`(리뷰별 토큰 리스트 534개), `top10_before`, `top10_after`(각각 단어 10개 리스트).
- **★ 지우면 안 되는 단어**: `크림·피부·로션·자외선·발림·끈적이` 는 **선크림 리뷰의 알맹이**입니다. 고빈도라고 해서 이런 단어까지 지우면 분석할 게 남지 않습니다 — 절대 불용어에 넣지 마세요.
- **주의**: 두 불용어 집합은 `|`(합집합)으로 합칠 수 있습니다. `Counter` 는 리뷰별 토큰 리스트를 `update` 로 이어서 세거나, 모든 토큰을 한 줄로 펼쳐 한 번에 셀 수 있어요. `most_common(n)` 은 `(단어, 빈도)` 튜플을 돌려주니 **단어만** 골라 담으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 일반 불용어만으로 한 번 토큰화해 빈도 상위를 눈으로 보고, 도메인에서 의미 없는 고빈도어만 골라
  불용어 집합에 더한 뒤 다시 토큰화해 전후 상위 단어를 비교한다.

세부구현:
1. df 의 text 를 하나씩 tokenize 해 리뷰별 토큰 리스트 tokens_general 을 만든다
2. 모든 토큰을 Counter 로 세어 상위 20개를 출력해 눈으로 확인한다
3. 지시된 14개 메타어로 domain_stopwords 집합을 만든다
4. 일반 불용어와 도메인 불용어를 합친 집합으로 다시 토큰화해 tokens 를 만든다
5. 전·후 Counter 의 상위 10개에서 단어만 뽑아 top10_before, `top10_after` 변수에 담고 나란히 출력한다
```

</details>

In [ ]:
tokens_general = [tokenize(text, stopwords) for text in df["text"]]
counter_before = Counter(word for token_list in tokens_general for word in token_list)
print("일반 불용어만 적용 — 빈도 top20:")
print(counter_before.most_common(20))

# 선크림 리뷰에서만 의미가 없는 고빈도어(리뷰 메타어) — 크림·피부·로션 등 알맹이는 남긴다
domain_stopwords = {"제품", "사용", "구매", "상품", "주문", "배송", "리뷰", "느낌", "생각", "쿠팡", "체험", "무료", "제공", "이벤트"}
all_stopwords = stopwords | domain_stopwords

tokens = [tokenize(text, all_stopwords) for text in df["text"]]
counter_after = Counter(word for token_list in tokens for word in token_list)

top10_before = [word for word, count in counter_before.most_common(10)]
top10_after = [word for word, count in counter_after.most_common(10)]
print("도메인 불용어 적용 전 top10:", top10_before)
print("도메인 불용어 적용 후 top10:", top10_after)

In [ ]:
# [자가채점]
assert domain_stopwords == {"제품", "사용", "구매", "상품", "주문", "배송", "리뷰", "느낌", "생각", "쿠팡", "체험", "무료", "제공", "이벤트"}
assert len(tokens) == 534
assert len(top10_before) == 10 and len(top10_after) == 10
assert "제품" in top10_before and "제품" not in top10_after
assert "사용" not in top10_after and "구매" not in top10_after
assert "피부" in top10_after and "크림" in top10_after
print("✅ 3단계 통과!")

### 해설 — 문제 1 · 3단계 — 도메인 불용어 구축
- **접근법**: 일반 불용어만으로는 부족합니다. **빈도 상위를 눈으로 보고** 이 도메인에서만 의미 없는 말(제품·구매·배송 같은 말)을 골라 추가한 뒤 다시 토큰화합니다.
- **흔한 실수**: 불용어를 너무 많이 넣어 **분석에 필요한 말까지 지우는 것**입니다. 뺄지 말지 애매하면 남겨 두고, 결과를 보며 조금씩 늘리는 편이 안전합니다.
- **대안**: `TfidfVectorizer(max_df=0.9)`는 문서빈도가 지나치게 높은 단어를 **후보 기준으로 제외**할 수 있습니다. 다만 의미를 이해하는 기능은 아니므로, 제외된 단어를 확인하고 도메인 판단과 함께 사용해야 합니다.

### 4단계 — 빈도 top15 막대그래프와 워드클라우드
3단계에서 정리한 `tokens` 로 **단어 빈도 top15 막대그래프**와 **워드클라우드**를 그리세요. **이 단계는 자가채점이 없습니다** — 아래 완성 그래프와 같은 모양이 나오면 됩니다.

1. `tokens` 의 전체 단어 빈도를 `Counter` 로 세어 상위 15개를 가져옵니다.
2. `fig, ax = plt.subplots()`로 Figure와 Axes를 만든 뒤 **가로 막대그래프**(`sns.barplot(x=counts, y=words, ax=ax, errorbar=None)`)로 그리고, 빈도가 큰 단어가 **위**에서부터 보이도록 `most_common(15)`의 순서를 그대로 사용합니다. 제목·축 이름은 `ax.set_*` 메서드로 답니다.
3. 다시 `fig, ax = plt.subplots()`로 새 Figure와 Axes를 만들고 **워드클라우드**를 그립니다. `WordCloud` 에 `font_path=FONT_PATH`, `background_color`, `width`·`height` 를 주고, **빈도 딕셔너리**로 그리는 메서드(`generate_from_frequencies`)에 상위 100개 단어의 `{단어: 빈도}` 딕셔너리를 넘기세요. `ax.imshow(...)`로 띄우고 `ax.axis("off")`로 축을 끕니다.

- **★ 필수**: `WordCloud(font_path=FONT_PATH, ...)` — 폰트를 안 넘기면 한글이 전부 □□□ 로 깨집니다.
- **주의**: 그래프를 그릴 때마다 `fig, ax = plt.subplots()`로 **새 Figure와 Axes**를 만들어야 두 그래프가 겹치지 않습니다. `Counter.most_common(n)` 은 `(단어, 빈도)` 튜플 리스트라 `dict(...)` 로 딕셔너리로 바꿀 수 있어요.

<details><summary>힌트</summary>

```text
접근방법:
- 빈도를 세어 상위 15개를 가로 막대로 그리고, 상위 100개 빈도 딕셔너리로 워드클라우드를 그린다.

세부구현:
1. tokens 의 모든 단어를 Counter 로 세고 most_common 으로 상위 15개를 가져온다
2. 단어와 빈도를 각각 리스트로 분리한다
3. fig, ax = plt.subplots()로 만든 뒤 sns.barplot(x=counts, y=words, ax=ax, errorbar=None)로 그린다
4. 다시 fig, ax = plt.subplots()로 만들고 WordCloud 결과를 ax.imshow로 표시한다
5. 상위 100개를 dict 로 바꿔 빈도 기반 생성 메서드에 넘긴다
6. imshow 로 띄우고 축을 끈 뒤 제목을 달아 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 두 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv3_q1_freq.png" width="560"/>

<img src="../../day11_텍스트전처리_분석/images/과제/lv3_q1_wordcloud.png" width="620"/>

In [ ]:
counter = Counter(word for token_list in tokens for word in token_list)
top15 = counter.most_common(15)
words = [word for word, count in top15]
counts = [count for word, count in top15]

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=counts, y=words, color="#4C72B0", ax=ax, errorbar=None)
ax.set_title("선크림 리뷰 단어 빈도 top15")
ax.set_xlabel("빈도")
ax.set_ylabel("단어")
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
wc = WordCloud(font_path=FONT_PATH, background_color="white", width=900, height=450)
wc = wc.generate_from_frequencies(dict(counter.most_common(100)))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("선크림 리뷰 워드클라우드")
plt.show()

### 해설 — 문제 1 · 4단계 — 빈도 top15와 워드클라우드
- **접근법**: `Counter.most_common(15)` 로 순위를 만들고, 같은 빈도 사전을 `generate_from_frequencies` 에 그대로 넘겨 워드클라우드를 그립니다.
- **흔한 실수**: 워드클라우드에 `font_path` 를 주지 않아 한글이 네모(두부)로 나오는 것입니다. 그리고 `generate()` 는 원문 문자열용이라, 빈도 사전에는 `generate_from_frequencies` 를 써야 합니다.
- **대안**: 막대그래프는 정확한 순위·수치를, 워드클라우드는 전체 인상을 보여 줍니다. 리포트에는 둘을 **함께** 넣어야 오해가 적습니다.

### 5단계 — TF-IDF 차이 기반 특징어 (만족 vs 불만)
이제 **만족 고객과 불만 고객이 서로 다르게 쓰는 말**을 찾습니다.

1. `tokens` 변수의 각 리뷰 토큰 리스트를 **공백으로 이어 붙인 문자열** 534개로 바꿔 `docs` 리스트 변수에 담습니다.
2. `TfidfVectorizer(min_df=5, token_pattern=r'\S+')` 로 `docs` 를 학습·변환해 행렬 `X` 를, 단어 목록을 `terms` 로 얻습니다. 토큰은 이미 공백으로 나뉘어 있으므로 공백이 아닌 덩어리를 그대로 한 단어로 봅니다.
3. 별점으로 두 그룹을 나눕니다 — **만족 = `rating` 4~5**(482건), **불만 = `rating` 1~3**(52건).
4. 각 그룹의 **평균 TF-IDF**(그룹에 속한 행들의 열 평균)를 구해 `mean_satisfied`·`mean_dissatisfied` 변수에 담습니다.
5. **차이 점수** = `mean_satisfied - mean_dissatisfied` 를 구해, 이 값이 **큰 순서로 10개** 단어를 `top_satisfied` 리스트 변수에, **작은 순서로 10개**(= 불만 쪽으로 치우친 단어)를 `top_dissatisfied` 리스트 변수에 담고 출력합니다.
6. 만족 특징어 중 `자극`·`백탁`·`끈적이`는 그 단어만으로 긍정 방향을 알 수 없습니다. `df['text'].str.contains('자극|백탁|끈적', na=False)`로 해당 원문을 찾아 `context_examples` DataFrame 변수에 5건 이상 담아 출력하고, 부정 표현(`없다`·`않다`)이 함께 쓰였는지 확인합니다.

- **요구 변수**: `docs`(문자열 534개), `X`(TF-IDF 행렬, 행 534), `terms`(단어 배열), `top_satisfied`·`top_dissatisfied`(각각 단어 문자열 **10개** 리스트), `context_examples`(조건에 맞는 원문 5건 이상 DataFrame).
- **왜 차이인가**: 각 그룹의 **평균 TF-IDF 상위**만 보면 양쪽 모두 `바르·크림·피부·썬크림` 처럼 **두 그룹이 공유하는 흔한 단어**가 올라와 구분이 안 됩니다. 두 평균의 **차이**를 보면 한쪽에서만 두드러지는 단어가 남습니다.
- **주의**: 희소 행렬은 `X.toarray()` 로 밀집 배열로 바꿔 평균을 낼 수 있습니다(534행이라 가볍습니다). 넘파이의 `argsort` 는 **오름차순** 정렬 인덱스를 돌려줍니다 — 큰 쪽 10개와 작은 쪽 10개를 각각 어떻게 잘라낼지 생각해 보세요. 점수 자체가 아니라 **단어 문자열**을 담습니다.
- **해석 주의**: unigram TF-IDF는 단어의 주변 문맥을 버립니다. `자극`이 만족 쪽에 올라도 `자극이 있다`인지 `자극이 없다`인지는 이 표만으로 알 수 없습니다. 또 불만 그룹은 52건으로 더 작으므로 차이 점수는 **특징어 후보**로 보고 원문 건수와 문맥을 함께 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 토큰을 공백으로 이어 문서 문자열로 만들고 TF-IDF 로 바꾼 뒤, 별점으로 두 그룹의 평균을 각각 내고
  두 평균의 차이가 큰 쪽·작은 쪽에서 단어를 뽑는다.

세부구현:
1. tokens 의 각 토큰 리스트를 공백으로 join 해 문자열 리스트를 `docs` 변수에 담는다
2. TfidfVectorizer 를 min_df=5 로 만들어 docs 를 fit_transform 하고 단어 목록을 `terms` 변수에 담는다
3. rating 이 4 이상인지로 만족 여부 마스크를 만든다
4. X 를 배열로 바꿔 만족 행들의 열 평균, 불만 행들의 열 평균을 각각 구한다
5. 두 평균의 차이를 구한다
6. 차이를 정렬해 큰 쪽 10개 단어를 `top_satisfied` 변수에, 작은 쪽 10개 단어를 `top_dissatisfied` 변수에 담는다
7. 두 리스트를 출력해 확인한다
8. 만족 리뷰 중 자극·백탁·끈적이 포함된 원문 5건 이상을 `context_examples` DataFrame 변수에 담아 출력한다
```

</details>

In [ ]:
docs = [" ".join(token_list) for token_list in tokens]
vectorizer = TfidfVectorizer(min_df=5, token_pattern=r'\S+')
X = vectorizer.fit_transform(docs)
terms = vectorizer.get_feature_names_out()
print("TF-IDF 행렬 shape:", X.shape)

is_satisfied = (df["rating"] >= 4).values
print("만족(4~5점):", int(is_satisfied.sum()), "불만(1~3점):", int((~is_satisfied).sum()))

dense = X.toarray()
mean_satisfied = dense[is_satisfied].mean(axis=0)
mean_dissatisfied = dense[~is_satisfied].mean(axis=0)
diff = mean_satisfied - mean_dissatisfied

order = np.argsort(diff)
top_satisfied = [terms[i] for i in order[::-1][:10]]
top_dissatisfied = [terms[i] for i in order[:10]]
print("만족 특징어 top10:", top_satisfied)
print("불만 특징어 top10:", top_dissatisfied)

context_mask = df["text"].astype(str).str.contains("자극|백탁|끈적", na=False)
context_examples = df.loc[is_satisfied & context_mask, ["rating", "text"]].head(8)
print("\n[만족 특징어의 원문 문맥 확인]")
display(context_examples)

In [ ]:
# [자가채점]
assert len(docs) == 534
assert X.shape[0] == 534
assert len(top_satisfied) == 10 and len(top_dissatisfied) == 10
assert not (set(top_satisfied) & set(top_dissatisfied))   # 두 그룹의 특징어는 겹치지 않는다
assert "크림" in top_satisfied and "자극" in top_satisfied
assert "두드러기" in top_dissatisfied and "포장" in top_dissatisfied
assert isinstance(context_examples, pd.DataFrame) and len(context_examples) >= 5
assert list(context_examples.columns) == ["rating", "text"]
assert (context_examples["rating"] >= 4).all()
assert context_examples["text"].astype(str).str.contains("자극|백탁|끈적", na=False).all()
print("✅ 5단계 통과!")

### 해설 — 문제 1 · 5단계 — TF-IDF 차이 기반 특징어
- **접근법**: 만족(고평점)·불만(저평점) 두 묶음의 TF-IDF 평균을 각각 구해 **차이(diff)** 를 냅니다. 양수면 만족 쪽, 음수면 불만 쪽에 더 연관된 특징어 후보입니다.
- **흔한 실수**: 빈도만으로 비교하면 양쪽에 다 흔한 말이 상위를 차지합니다. **차이를 봐야** 한쪽에만 치우친 말이 드러납니다. 또 `argsort` 는 오름차순이라 상위를 뽑으려면 부호를 뒤집어야 합니다.
- **대안**: unigram은 부정 표현과 주변 문맥을 잃을 수 있으므로 `context_examples`처럼 원문을 확인하고, 뒤 단계의 bigram 결과와 함께 해석해야 합니다.

### 6단계 — 별점별 키워드 변화 추적 (히트맵)
앞 단계에서 리뷰를 만족·불만 **두 덩어리**로만 나눴습니다. 이번엔 **별점 1~5 각각**이 어떤 말을 쓰는지 보고, **별점이 낮아질수록 어떤 단어가 등장하는지** 를 추적합니다(실무에서 자주 하는 분석).

1. 별점(`rating`)마다 그 별점 리뷰들의 **평균 TF-IDF**를 구해, 값이 큰 **상위 10개 단어**를 `rating_keywords` 변수에 담으세요 — **키 = 별점(정수), 값 = 단어 10개 리스트** 인 딕셔너리입니다. (5단계의 `dense`·`terms` 를 그대로 재사용하면 됩니다.)
2. 히트맵용 표를 만듭니다. 별점별 **상위 5개** 단어를 모두 모아 **중복을 없앤** 단어 목록을 만들고(등장 순서 유지), **행 = 별점(5점→1점), 열 = 그 단어들**, 값 = 그 별점의 그 단어 **평균 TF-IDF** 인 DataFrame을 `rating_table` 변수에 담으세요.
3. `fig, ax = plt.subplots()`로 Figure와 Axes를 만든 뒤 `sns.heatmap(..., ax=ax)`으로 그립니다 — `annot=True`, `fmt=".2f"`, `cmap="YlOrRd"`, `annot_kws={"size": 7}`를 주고 `ax.set_title`로 제목을 답니다.

- **요구 변수**: `rating_keywords`(키 = 별점 `1`~`5`, 값 = 단어 10개 리스트인 딕셔너리), `rating_table`(별점×단어 평균 TF-IDF DataFrame).
- **데이터 주의**: 이 리뷰의 별점 분포는 **5점 440 · 4점 42 · 3점 22 · 2점 18 · 1점 12** 로 **5점에 크게 치우쳐** 있습니다. 다만 가장 적은 1점도 12건이라 **다섯 별점 모두 포함**해 분석합니다 (리뷰가 몇 건뿐인 별점은 보통 제외하지만, 여기서는 그럴 필요가 없습니다). 낮은 별점은 표본이 적으니 **경향으로만** 읽으세요.
- **그래프 채점**: 이 단계의 **그래프는 자가채점이 없습니다** — 아래 완성 그래프처럼 그리면 됩니다. 자가채점은 `rating_keywords`와 `rating_table`의 **형태·값 범위**를 확인합니다.
- **주의**: `dense[별점 마스크].mean(axis=0)` 으로 그 별점의 평균 TF-IDF 벡터를 얻습니다. 중복 제거는 순서가 유지되도록 *이미 담긴 단어면 건너뛰기* 방식으로 하세요(`set` 은 순서가 흐트러집니다).

<details><summary>힌트</summary>

```text
접근방법:
- 별점마다 그 별점 리뷰들의 평균 TF-IDF 를 구해 상위 단어를 뽑고, 상위 5개씩을 모아 만든 단어 목록으로
  별점 x 단어 표를 만들어 색으로 칠한다.

세부구현:
1. 결과를 담을 빈 딕셔너리를 만든다
2. 별점을 오름차순으로 훑으며 그 별점 행만 고르는 마스크를 만든다
3. 그 행들의 열 평균을 구해 값이 큰 상위 10개 단어를 `rating_keywords` 변수에 담는다
4. 별점을 높은 순으로 훑으며 각 별점 상위 5개 단어를 아직 없을 때만 목록에 추가한다
5. 별점(행) x 그 단어들(열) 의 평균 TF-IDF 값을 모아 `rating_table` DataFrame 변수에 담는다
6. fig, ax = plt.subplots()로 만들고 sns.heatmap(..., ax=ax)으로 그린 뒤 제목을 달아 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv3_q1_rating_heatmap.png" width="700"/>

In [ ]:
rating_keywords = {}
for rating in sorted(df["rating"].unique()):
    mask = (df["rating"] == rating).values
    mean_scores = dense[mask].mean(axis=0)
    rating_keywords[int(rating)] = [terms[i] for i in np.argsort(mean_scores)[::-1][:10]]
    print(rating, "점 상위 키워드:", rating_keywords[int(rating)])

columns = []
for rating in sorted(df["rating"].unique(), reverse=True):
    for word in rating_keywords[int(rating)][:5]:
        if word not in columns:
            columns.append(word)

rows = []
index = []
for rating in sorted(df["rating"].unique(), reverse=True):
    mask = (df["rating"] == rating).values
    mean_scores = dense[mask].mean(axis=0)
    rows.append([mean_scores[list(terms).index(word)] for word in columns])
    index.append(f"{int(rating)}점")
rating_table = pd.DataFrame(rows, index=index, columns=columns)

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(rating_table, annot=True, fmt=".2f", cmap="YlOrRd", annot_kws={"size": 7}, ax=ax)
ax.set_title("별점별 상위 키워드 평균 TF-IDF")
plt.show()

In [ ]:
# [자가채점]
assert set(rating_keywords.keys()) == set(int(r) for r in df["rating"].unique())
assert len(rating_keywords[5]) == 10
assert all(len(words) == 10 for words in rating_keywords.values())
assert "두드러기" in rating_keywords[1]   # 1점 리뷰에서만 두드러지는 말
assert "크림" in rating_keywords[5]
assert isinstance(rating_table, pd.DataFrame)
assert rating_table.shape == (5, 15)
assert list(rating_table.index) == ["5점", "4점", "3점", "2점", "1점"]
assert rating_table.columns.is_unique
assert (rating_table.to_numpy() >= 0).all()
print("✅ 6단계 통과!")

### 해설 — 문제 1 · 6단계 — 별점별 키워드 변화 (히트맵)
- **접근법**: 별점별로 키워드 점수를 행렬로 만들어 히트맵으로 그립니다. **점수가 어느 별점에서 솟는지**를 색으로 한눈에 보는 것이 목적입니다.
- **흔한 실수**: 행마다 스케일이 달라 색이 한쪽으로 쏠리는 것입니다. 필요하면 행 기준으로 정규화하세요. 그리고 축 라벨이 한글이면 폰트 설정을 빠뜨리지 마세요.
- **대안**: 별점이 5단계라 칸이 적으면 `annot=True` 로 숫자를 함께 찍는 편이 읽기 좋습니다.

### 7단계 — 리포트 표 만들기와 CSV 저장
5단계의 두 특징어 목록을 하나의 **리포트 표** `report` 로 합쳐 저장합니다.

| 컬럼 | 내용 |
| --- | --- |
| `group` | 그룹 이름 — 만족 특징어는 `"만족"`, 불만 특징어는 `"불만"` |
| `rank` | 그 그룹 안에서의 순위 — **1부터 10까지** |
| `word` | 특징어 단어 |

- **요구 변수**: `report`(위 3개 컬럼을 가진 DataFrame, **20행** = 만족 10행 + 불만 10행). 컬럼명·그룹 이름은 표와 **똑같이** 쓰세요.
- **요구사항**: `output` 폴더가 없을 수 있으니 `os.makedirs(..., exist_ok=True)` 로 먼저 만든 뒤 `output/text_report.csv` 로 저장하고(인덱스 없이), `display(report)` 로 표를 확인하세요.
- **주의**: `os` 는 아직 import 하지 않았으니 이 셀에서 `import os` 를 먼저 하세요. 순위는 1부터 시작합니다(0 아님) — `enumerate` 의 시작값을 바꿀 수 있어요.

<details><summary>힌트</summary>

```text
접근방법:
- 두 특징어 리스트를 그룹 이름·순위와 함께 행 딕셔너리로 펼쳐 표로 만들고, 폴더를 만든 뒤 CSV 로 내보낸다.

세부구현:
1. os 를 임포트하고 행을 담을 리스트를 만든다
2. 만족 리스트를 순위와 함께 훑으며 group·rank·word 딕셔너리를 추가한다
3. 불만 리스트도 같은 방식으로 추가한다
4. 행 리스트로 DataFrame report 를 만든다
5. makedirs 로 저장 폴더를 만들고 to_csv 로 인덱스 없이 저장한다
6. display 로 표를 확인한다
```

</details>

In [ ]:
import os
rows = []
for rank, word in enumerate(top_satisfied, start=1):
    rows.append({"group": "만족", "rank": rank, "word": word})
for rank, word in enumerate(top_dissatisfied, start=1):
    rows.append({"group": "불만", "rank": rank, "word": word})
report = pd.DataFrame(rows)

os.makedirs("output", exist_ok=True)
report.to_csv("output/text_report.csv", index=False)
print("저장 완료:", "output/text_report.csv")
display(report)

In [ ]:
# [자가채점]
import os
assert set(report.columns) == {"group", "rank", "word"}
assert len(report) == 20
assert set(report["group"]) == {"만족", "불만"}
assert sorted(report.loc[report["group"] == "만족", "rank"]) == list(range(1, 11))
assert sorted(report.loc[report["group"] == "불만", "rank"]) == list(range(1, 11))
assert report.loc[report["group"] == "만족"].sort_values("rank")["word"].tolist() == top_satisfied
assert report.loc[report["group"] == "불만"].sort_values("rank")["word"].tolist() == top_dissatisfied
saved_report = pd.read_csv("output/text_report.csv")
assert saved_report.equals(report.reset_index(drop=True))
print("✅ 7단계 통과!")

### 해설 — 문제 1 · 7단계 — 리포트 표와 CSV 저장
- **접근법**: 지금까지 구한 지표를 하나의 DataFrame 으로 모아 `to_csv(index=False)` 로 저장합니다. **분석의 산출물을 파일로 남기는 것**까지가 한 사이클입니다.
- **흔한 실수**: `index=False` 를 빠뜨려 이름 없는 인덱스 열이 붙는 것, 그리고 한글이 깨지는 인코딩 문제입니다. 필요하면 `encoding='utf-8-sig'` 로 저장하면 엑셀에서도 잘 열립니다.
- **대안**: 표를 만들 때 컬럼 이름을 한글로 두면 그대로 리포트에 붙여 쓸 수 있어 편합니다.

### 인사이트 — 소비자 관심사 읽기 (서술)
완성한 `report` 의 만족·불만 특징어와 6단계의 **별점별 히트맵**을 근거로, 이 선크림 리뷰어들이 **무엇에 만족하고 무엇에 불만인지**, 그리고 **별점이 낮아질수록 어떤 말이 등장하는지** 를 **2~3문장**으로 서술하세요. 단, `자극`·`백탁`·`끈적이`처럼 긍정·부정 방향이 문맥에 따라 달라지는 단어는 5단계의 `context_examples` 원문을 근거로 해석하세요.

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 정답은 여럿)**

만족 쪽 특징어 후보에는 `크림·피부·자극·부드럽·백탁·끈적이·만족·자외선·성분`처럼 **제품의 사용감**과 관련된 말이 보인다. 다만 unigram만으로는 방향을 알 수 없어 원문을 확인했으며, 실제 만족 리뷰의 `자극이 느껴지지 않았다`·`백탁이 없다`·`끈적이지 않다` 같은 문맥을 근거로 *자극·백탁·끈적임이 적고 부드럽게 발리는 점*을 만족 요인 후보로 해석할 수 있다. 반면 불만 특징어는 `두드러기·올라오·땡기·심하` 같은 **피부 트러블**과 `박스·포장·버리` 같은 **배송·포장 상태**로 갈린다. 별점별 히트맵을 보면 이 흐름이 더 뚜렷하다 — **5점**에서는 `크림·피부·끈적이` 같은 사용감 어휘가 고르게 높지만, **별점이 내려갈수록** `얼굴`·`눈물`(눈 시림)이 커지다가 **1점에서 `두드러기`·`심하`·`피지오겔`(대체품 언급)이 급격히 튀어오른다**. 정리하면 이 표본에서는 소비자 관심사가 **피부 자극·트러블, 발림성, 배송·포장 상태**와 연결되어 보인다. 다만 불만 그룹과 낮은 별점 그룹은 표본이 작으므로 이를 곧바로 전체 고객의 개선 우선순위로 확정하지 말고, 원문 건수와 추가 표본에서 같은 경향이 반복되는지 확인해야 한다.

**해설 — 왜 차이 기반인가**

- 두 그룹의 **평균 TF-IDF 상위**를 그냥 보면 만족·불만 양쪽 모두 `바르·크림·피부·썬크림` 이 올라온다. 두 그룹이 **공유하는 흔한 단어**라 구분에 쓸모가 없다(공유 노이즈).
- 두 평균의 **차이**를 보면 이 공유 노이즈가 서로 상쇄돼 사라지고, **한쪽에서만 두드러지는 단어**(`두드러기`·`포장` 등)가 남는다. 그래서 그룹을 비교할 때는 절대 점수가 아니라 **차이**를 본다.
- 하지만 차이는 **연관 후보 점수**일 뿐 단어의 긍·부정 문맥이나 인과를 복원하지 않습니다. `자극`처럼 부정 표현과 함께 쓰이는 단어는 반드시 원문을 확인하고, 작은 그룹의 결과는 경향으로만 읽습니다.
- **흔한 실수 2가지**: 1) `argsort` 결과를 뒤집지 않아 만족·불만 목록이 뒤바뀌는 것 (오름차순이 기본), 2) 도메인 불용어를 3단계에서 적용하지 않은 채 TF-IDF 를 돌려 `제품·사용` 같은 메타어가 특징어로 올라오는 것.

## 2. 연관어 탐색기
**배경**: 같은 선크림 리뷰에서 이번엔 **어떤 단어들이 붙어 다니는지** 를 찾습니다. 바로 앞뒤로 이어지는 **두 단어 묶음(bigram)** 과, 한 리뷰 안에 **함께 등장하는 단어쌍(동시 출현)** 을 세어 제품의 어떤 속성이 함께 이야기되는지 확인하는 프로그램입니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 문제 1의 전처리를 재사용해 `docs` 534개 |
| 2단계 | bigram(`ngram_range=(2, 2)`, `min_df=3`) 상위 10개 `bigram_top10` |
| 3단계 | 동시 출현 행렬(`binary=True`, `min_df=10`, `max_df=0.85`) → 상위 단어쌍 30개 `pairs` |
| 4단계 | 동시 출현 히트맵(상위 15개 단어 부분행렬, 완성 그래프처럼) |
| 5단계 | 동시 출현 **네트워크**(제공 함수 호출, 완성 그래프처럼) |
| 인사이트 | 어떤 단어가 함께 등장하는지·그것이 제품의 어떤 속성인지 2~3문장 서술 |

### 1단계 — 전처리된 문서 준비
문제 1에서 만든 **전처리 함수와 불용어**를 그대로 재사용합니다. 리뷰 534건을 `tokenize(text, 일반 불용어 + 도메인 불용어)` 로 토큰화하고, 각 리뷰의 토큰을 **공백으로 이어 붙인 문자열** 리스트 `docs` 를 만드세요(534개). 그리고 `docs[0]` 의 앞부분을 출력해 확인하세요.

- **요구 변수**: `docs`(공백으로 이어 붙인 문자열 리스트, 534개).
- **주의**: 문제 1의 3단계에서 만든 `tokens` 를 그대로 이어 써도 되고, 다시 토큰화해도 됩니다 — **단, 도메인 불용어(`제품·사용·구매`…)까지 적용된 토큰**이어야 뒤 결과가 지문의 값과 맞습니다. 이 단계 이후 `CountVectorizer` 는 공백을 기준으로 단어를 쪼갭니다.

<details><summary>힌트</summary>

```text
접근방법:
- 도메인 불용어까지 적용한 리뷰별 토큰 리스트를 공백으로 이어 붙여 문서 문자열 리스트를 만든다.

세부구현:
1. 일반 불용어와 도메인 불용어를 합친 집합을 준비한다
2. df 의 text 를 하나씩 tokenize 해 리뷰별 토큰 리스트를 얻는다
3. 각 토큰 리스트를 공백으로 join 해 `docs` 변수에 담는다
4. 문서 수와 첫 문서의 앞부분을 출력해 확인한다
```

</details>

In [ ]:
all_stopwords = stopwords | domain_stopwords
tokens = [tokenize(text, all_stopwords) for text in df["text"]]
docs = [" ".join(token_list) for token_list in tokens]
print("문서 수:", len(docs))
print("첫 문서 앞부분:", docs[0][:60])

In [ ]:
# [자가채점]
assert len(docs) == 534
assert all(isinstance(doc, str) for doc in docs)
print("✅ 1단계 통과!")

### 해설 — 문제 2 · 1단계 — 전처리된 문서 준비
- **접근법**: 문제 1 에서 만든 토큰 리스트를 **공백으로 이어 붙인 문자열 목록**으로 바꿉니다. `CountVectorizer` 가 문자열을 입력으로 받기 때문입니다.
- **흔한 실수**: 토큰 리스트를 그대로 넣는 것입니다. 또 `token_pattern` 을 기본값으로 두면 한 글자 토큰이 버려지므로 `token_pattern=r'\S+'` 로 **공백 기준**임을 알려 줘야 합니다.
- **대안**: 이미 토큰화가 끝났다면 `analyzer=lambda x: x` 로 리스트를 그대로 받게 하는 방법도 있습니다.

### 2단계 — 두 단어 묶음(bigram) 상위 10개
`CountVectorizer` 로 **바로 이웃한 두 단어 묶음**을 세어 가장 많이 등장하는 10개를 뽑으세요.

1. `CountVectorizer(ngram_range=(2, 2), min_df=3, token_pattern=r'\S+')` 로 `docs` 를 학습·변환합니다. (`ngram_range=(2, 2)` = 두 단어 묶음만, `min_df=3` = 3개 미만의 문서에만 나오는 묶음은 버림.)
2. 묶음 이름은 `get_feature_names_out()`, 각 묶음의 **전체 등장 횟수**는 변환된 행렬을 **열 방향으로 합**해서 얻습니다.
3. 등장 횟수가 큰 순서로 10개를 `(묶음, 횟수)` 튜플 `bigram_top10` 리스트에 담고 출력하세요. 횟수는 **파이썬 정수(`int`)** 로 바꿔 담습니다.

- **요구 변수**: `bigram_top10`(`(묶음 문자열, 정수 횟수)` 튜플 **10개** 리스트, 횟수 내림차순).
- **요구사항**: 1위는 `"크림 바르"`(119회), 그다음이 `"자외선 차단"`(107회) 입니다.
- **주의**: 희소 행렬의 열 합은 `X.sum(axis=0)` 으로 얻은 뒤 `np.asarray(...).ravel()` 로 1차원 배열로 펴서 다루면 편합니다. 묶음 문자열은 두 단어가 공백으로 이어진 형태입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 단어 묶음만 세는 카운트 벡터라이저로 문서를 변환하고, 묶음별 총 등장 횟수를 구해 상위 10개를 뽑는다.

세부구현:
1. CountVectorizer 를 ngram_range=(2, 2), min_df=3 으로 만들어 docs 를 fit_transform 한다
2. get_feature_names_out 으로 묶음 이름 배열을 얻는다
3. 행렬을 열 방향으로 합해 묶음별 총 횟수 배열을 만든다
4. 횟수를 내림차순으로 정렬한 인덱스를 구한다
5. 앞 10개 인덱스로 (묶음, 정수 횟수) 튜플 리스트를 만들어 `bigram_top10` 변수에 담고 출력한다
```

</details>

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2), min_df=3, token_pattern=r'\S+')
bigram_matrix = bigram_vectorizer.fit_transform(docs)
bigram_terms = bigram_vectorizer.get_feature_names_out()
bigram_counts = np.asarray(bigram_matrix.sum(axis=0)).ravel()

order = np.argsort(bigram_counts)[::-1][:10]
bigram_top10 = [(bigram_terms[i], int(bigram_counts[i])) for i in order]
print("bigram top10:")
for phrase, count in bigram_top10:
    print(f"{phrase}: {count}")

In [ ]:
# [자가채점]
assert len(bigram_top10) == 10
bigram_dict = dict(bigram_top10)
assert bigram_top10[0][0] == "크림 바르"
assert bigram_dict["크림 바르"] == 119
assert bigram_dict["자외선 차단"] == 107
assert all(isinstance(count, int) for phrase, count in bigram_top10)
print("✅ 2단계 통과!")

### 해설 — 문제 2 · 2단계 — bigram 상위 10개
- **접근법**: `CountVectorizer(ngram_range=(2, 2))` 로 이웃한 두 낱말 묶음을 셉니다. 낱말 하나씩 셀 때는 안 보이던 **연결 관계**가 드러납니다.
- **흔한 실수**: `min_df` 를 기본값(1)으로 두면 한 번만 나온 희귀 묶음이 잔뜩 섞입니다. 반대로 너무 높이면 의미 있는 표현까지 사라지니, 문서 수를 보고 정하세요.
- **대안**: `ngram_range=(1, 2)` 로 두면 낱말과 두 낱말 묶음을 **함께** 셀 수 있습니다.

### 3단계 — 동시 출현 행렬과 상위 단어쌍
이번엔 **한 리뷰 안에 함께 등장한** 단어쌍을 셉니다(붙어 있지 않아도 됩니다).

1. `CountVectorizer(binary=True, min_df=10, max_df=0.85, token_pattern=r'\S+')` 로 `docs` 를 변환해 행렬 `X` 를 얻습니다.
   - `binary=True` — *몇 번 나왔나* 가 아니라 **그 문서에 나왔나(0/1)** 만 기록합니다.
   - `min_df=10` — **너무 드문 단어**(10개 미만의 리뷰에만 나오는 단어)를 버립니다.
   - `max_df=0.85` — **너무 흔한 단어**(전체 리뷰의 85% 를 넘게 나오는 단어)를 버립니다. 모든 리뷰에 나오는 단어는 아무하고나 함께 등장해 연관 분석을 흐리기 때문입니다. (이 데이터에서는 가장 흔한 단어도 63% 수준이라 실제로 걸러지는 단어는 없지만, 다른 데이터에서 흔한 단어가 결과를 뒤덮는 것을 막아 주는 **안전장치**입니다.)
2. 단어 목록을 `terms` 로 얻고, **단어×단어 동시 출현 행렬** `M` 을 만듭니다 — `X` 의 **전치(`.T`)와 `X` 의 행렬곱**을 `.toarray()` 로 밀집 배열로 바꾸면 됩니다. 이 행렬의 `M[i][j]` 는 *단어 i 와 단어 j 가 함께 나온 리뷰 수* 입니다.
3. 대각선(`M[i][i]`)은 자기 자신과의 카운트라 무의미하니 `np.fill_diagonal` 로 0 으로 만듭니다.
4. 같은 쌍이 두 번 세어지지 않도록 **상삼각**(`i < j`)에서만 **상위 30쌍**을 뽑아, `(단어1, 단어2, 정수 횟수)` 튜플 `pairs` 리스트에 담고(횟수 내림차순) **앞 10쌍을 출력**하세요. (30쌍을 만드는 이유: 5단계 네트워크 그림에 25쌍을 넘겨 쓰기 때문입니다.)

- **요구 변수**: `M`(정사각 동시 출현 행렬, 대각선 0), `pairs`(`(단어1, 단어2, 정수 횟수)` 튜플 **30개** 리스트, 횟수 내림차순).
- **요구사항**: `M` 은 **단어 수 × 단어 수 의 정사각 행렬**(이 데이터에선 약 230개 단어)이고, 가장 많이 함께 나온 쌍은 `바르`·`크림` (**188개 리뷰**) 입니다.
- **주의**: 상삼각 인덱스는 `np.triu_indices_from` 에 `k=1`(대각선 제외)을 주어 얻을 수 있습니다. 행렬곱은 `@` 연산자를 씁니다. 한 쌍의 두 단어는 순서를 신경 쓰지 않아도 됩니다. 횟수는 **파이썬 정수(`int`)** 로 바꿔 담으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문서마다 단어의 등장 여부만 0/1 로 기록한 행렬을 만들고, 그 전치와 곱해 단어끼리 함께 나온 횟수를
  구한 뒤 대각선을 지우고 위쪽 삼각에서 큰 값 30개를 뽑는다.

세부구현:
1. CountVectorizer 를 binary=True, min_df=10, max_df=0.85 로 만들어 docs 를 fit_transform 한다
2. get_feature_names_out 으로 단어 배열 terms 를 얻는다
3. 행렬의 전치와 행렬 자신을 곱하고 밀집 배열로 바꿔 M 을 만든다
4. fill_diagonal 로 대각선을 0 으로 만든다
5. 대각선을 제외한 상삼각 인덱스를 구하고 그 위치의 값들을 가져온다
6. 값이 큰 순서로 30개를 골라 (단어1, 단어2, 정수 횟수) 튜플로 담아 pairs 에 넣는다
7. 앞 10쌍을 출력해 확인한다
```

</details>

In [ ]:
co_vectorizer = CountVectorizer(binary=True, min_df=10, max_df=0.85, token_pattern=r'\S+')
X = co_vectorizer.fit_transform(docs)
terms = co_vectorizer.get_feature_names_out()

M = (X.T @ X).toarray()
np.fill_diagonal(M, 0)
print("동시 출현 행렬 shape:", M.shape)

rows_idx, cols_idx = np.triu_indices_from(M, k=1)
values = M[rows_idx, cols_idx]
order = np.argsort(values)[::-1][:30]
pairs = [(terms[rows_idx[i]], terms[cols_idx[i]], int(values[i])) for i in order]
print("함께 등장한 단어쌍 top10:")
for word1, word2, count in pairs[:10]:
    print(f"{word1} + {word2}: {count}")

In [ ]:
# [자가채점]
assert M.shape[0] == M.shape[1]   # 정사각 행렬
assert (M == M.T).all()           # 대칭 행렬
assert (np.diag(M) == 0).all()     # 대각선 전체는 0
assert len(pairs) == 30
assert {pairs[0][0], pairs[0][1]} == {'바르', '크림'}
assert pairs[0][2] == 188
assert all(isinstance(count, int) for word1, word2, count in pairs)
assert pairs[0][2] >= pairs[-1][2]   # 횟수 내림차순
print("✅ 3단계 통과!")

### 해설 — 문제 2 · 3단계 — 동시 출현 행렬
- **접근법**: 문서-단어 행렬을 이진화한 뒤 `X.T @ X` 로 곱하면 **두 단어가 같은 문서에 함께 나온 횟수** 행렬이 됩니다. 대각선(자기 자신)은 0으로 지웁니다.
- **흔한 실수**: 이진화를 건너뛰면 한 문서에 여러 번 나온 단어가 과대 계산됩니다. 그리고 대각선을 남겨 두면 자기 자신이 1위가 되어 상위 쌍이 엉망이 됩니다.
- **대안**: 행렬은 대칭이라 상삼각만 보면 충분합니다. `np.triu` 로 중복 쌍을 제거하면 순위가 깔끔해집니다.

### 4단계 — 동시 출현 히트맵
3단계의 동시 출현 행렬 `M` 을 **색으로** 봅니다. 단어가 200개를 넘어 전부 그리면 읽을 수 없으니, **함께 등장한 횟수의 합이 큰 상위 15개 단어**만 골라 **15×15 부분행렬**을 히트맵으로 그리세요. **이 단계는 자가채점이 없습니다** — 아래 완성 그래프와 같은 모양이 나오면 됩니다.

1. 각 단어의 **행 합**(`M.sum(axis=1)`)이 큰 순서로 15개 단어의 인덱스를 고릅니다.
2. 그 인덱스로 `M` 의 **행과 열을 동시에** 잘라 15×15 부분행렬을 만들고, 단어 이름을 인덱스·컬럼으로 가진 DataFrame을 `sub_df` 변수에 담습니다(축에 한글 단어가 찍히도록).
3. `fig, ax = plt.subplots()`로 Figure와 Axes를 만든 뒤 `sns.heatmap(..., ax=ax)`으로 그립니다 — `annot=True`, `fmt="d"`, `cmap="YlOrRd"`, `annot_kws={"size": 7}`를 주고 제목을 답니다.

- **주의**: 부분행렬은 넘파이의 `np.ix_(...)` 로 행·열 인덱스를 함께 넘기면 한 번에 잘라집니다. 그래프를 그릴 때마다 `fig, ax = plt.subplots()`로 **새 Figure와 Axes**를 만드세요.

<details><summary>힌트</summary>

```text
접근방법:
- 단어별 동시 출현 총합이 큰 15개를 골라 그 단어들끼리의 부분행렬을 만들고 색으로 칠한다.

세부구현:
1. M 을 행 방향으로 합해 단어별 총 동시 출현 횟수를 구한다
2. 그 값이 큰 순서로 15개 단어의 인덱스를 고른다
3. np.ix_ 로 행·열을 함께 잘라 15x15 부분행렬을 만든다
4. 단어 이름을 인덱스·컬럼으로 주어 sub_df DataFrame 변수에 담는다
5. fig, ax = plt.subplots()로 만들고 sns.heatmap(..., ax=ax)으로 그린 뒤 제목을 달아 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv3_q2_heatmap.png" width="640"/>

In [ ]:
totals = M.sum(axis=1)
top15_idx = np.argsort(totals)[::-1][:15]
top15_words = [terms[i] for i in top15_idx]

sub_matrix = M[np.ix_(top15_idx, top15_idx)]
sub_df = pd.DataFrame(sub_matrix, index=top15_words, columns=top15_words)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sub_df, annot=True, fmt="d", cmap="YlOrRd", annot_kws={"size": 7}, ax=ax)
ax.set_title("단어 동시 출현 히트맵 (상위 15개 단어)")
plt.show()

### 해설 — 문제 2 · 4단계 — 동시 출현 히트맵
- **접근법**: 상위 단어들만 골라 부분 행렬을 만들고 `sns.heatmap(annot=True)` 로 그립니다. **어떤 쌍이 유독 진한지**를 보는 것이 목적입니다.
- **흔한 실수**: 전체 단어를 다 넣어 칸이 수백 개가 되는 것입니다. 라벨이 겹쳐 아무것도 못 읽습니다 — 상위 10~15개로 좁히세요.
- **대안**: 값의 범위가 넓으면 `np.log1p` 로 로그를 씌워 그리면 약한 관계도 함께 보입니다.

### 5단계 — 동시 출현 네트워크 그리기
히트맵이 *숫자 표를 색으로* 보는 방식이라면, **네트워크**는 단어를 **점(노드)** 으로, 함께 등장한 관계를 **선(엣지)** 으로 그려 **어떤 단어들이 한 덩어리로 뭉치는지**를 한눈에 보여 줍니다.

그리는 함수는 아래 `# [제공 코드]` 셀로 드립니다 — **내용은 이해하지 않아도 되고, 호출만** 하세요. 먼저 제공 코드 셀을 실행해 함수를 등록한 뒤, 답안 셀에서 3단계의 `pairs` 를 넘겨 호출하면 됩니다.

| 인자 | 뜻 |
| --- | --- |
| `pairs` | 3단계에서 만든 `(단어1, 단어2, 동시출현횟수)` 튜플 리스트 |
| `top_n` | 상위 몇 쌍을 그릴지 — **25** 로 주세요 |
| `title` | 그림 제목 — `"선크림 리뷰 동시 출현 네트워크"` |

함수의 반환값인 `networkx.Graph` 객체는 `graph` 변수에 담고, `graph.number_of_nodes()`와 `graph.number_of_edges()`로 노드·엣지 수를 출력하세요.

- **그림 읽는 법**: **노드 크기 = 연결 강도**(그 단어가 다른 단어들과 함께 등장한 횟수의 합), **엣지 두께 = 동시 출현 횟수**. 굵은 선으로 촘촘히 묶인 단어들이 **함께 이야기되는 주제 덩어리**입니다.
- **이 단계는 자가채점이 없습니다** — 아래 완성 그래프와 같은 모양이 나오면 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공된 네트워크 함수에 3단계의 단어쌍 리스트를 넘겨 호출한다.

세부구현:
1. 제공 코드 셀을 먼저 실행해 그리기 함수를 등록한다
2. 그 함수에 pairs 를 넘기고 top_n 과 title 을 지정해 호출한 반환값을 graph 변수에 담는다
3. graph.number_of_nodes()와 graph.number_of_edges()를 출력한다
```

</details>

In [ ]:
# [제공 코드] 동시 출현 네트워크를 그려 주는 함수입니다 — 내용은 이해하지 않아도 됩니다. 호출만 하세요.
import networkx as nx

def draw_cooccurrence_network(pairs, top_n=25, title='동시 출현 네트워크',
                              node_color='#6aa9e9', save_path=None):
    """pairs = [(단어1, 단어2, 동시출현횟수), ...] 를 네트워크로 그립니다."""
    G = nx.Graph()
    for w1, w2, cnt in pairs[:top_n]:
        G.add_edge(w1, w2, weight=int(cnt))
    strength = {n: sum(d['weight'] for _, _, d in G.edges(n, data=True)) for n in G.nodes()}
    lo, hi = min(strength.values()), max(strength.values())
    rng = (hi - lo) or 1
    sizes = [300 + (strength[n] - lo) / rng * 2200 for n in G.nodes()]
    mx = max(d['weight'] for _, _, d in G.edges(data=True))
    widths = [0.5 + G[u][v]['weight'] / mx * 4 for u, v in G.edges()]
    pos = nx.spring_layout(G, k=0.7, seed=42)
    fig, ax = plt.subplots(figsize=(11, 8))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.35, edge_color='#999999', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_color, alpha=0.9,
                           edgecolors='white', linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_family=KOREAN_FONT, font_size=11, ax=ax)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=110, bbox_inches='tight')
    plt.show()
    return G

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv3_q2_network.png" width="700"/>

In [ ]:
graph = draw_cooccurrence_network(pairs, top_n=25,
                                  title="선크림 리뷰 동시 출현 네트워크")
print("노드 수:", graph.number_of_nodes(), "엣지 수:", graph.number_of_edges())

### 해설 — 문제 2 · 5단계 — 동시 출현 네트워크
- **접근법**: 단어를 점(노드), 함께 나온 관계를 선(엣지)으로 그립니다. 히트맵이 **표**라면 네트워크는 **지도** — 어떤 단어가 중심에 있는지가 눈에 들어옵니다.
- **흔한 실수**: 쌍을 너무 많이 그려 선이 뒤엉키는 것입니다. 상위 15~20쌍으로 줄여야 읽힙니다. 노드 위치는 실행마다 달라지므로 재현이 필요하면 `seed` 를 고정하세요.
- **대안**: 선 굵기를 동시 출현 횟수에 비례시키면 강약이 함께 보입니다. 다만 네트워크는 **인상**을 주는 그림이라, 정확한 수치는 표로 함께 제시해야 합니다.

### 인사이트 — 함께 등장하는 말 읽기 (서술)
`bigram_top10`·`pairs`·히트맵·네트워크를 근거로, **어떤 단어들이 함께 등장하는지** 와 그것이 이 제품의 **어떤 속성**을 말하는지 **2~3문장**으로 서술하세요. 네트워크에서는 **노드 크기 = 연결 강도**, **엣지 두께 = 동시 출현 횟수** 라는 점을 이용해 읽으세요.

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 정답은 여럿)**

bigram 상위에는 `자외선 차단`(107회)·`부드럽 발리`(47회)처럼 **제품의 기능(자외선 차단)과 사용감(부드럽게 발림)** 을 가리키는 굳어진 표현이 올라온다. 동시 출현에서도 `바르`+`크림`(188), `끈적이`+`피부`(89), `끈적이`+`크림`(88) 처럼 **바르는 행위·피부 느낌·끈적임**이 한 리뷰 안에서 함께 이야기되고, 히트맵을 보면 `크림·피부·끈적이·발림·백탁` 이 서로 촘촘히 연결된 한 덩어리를 이룬다. 네트워크에서도 **노드가 가장 큰**(=연결 강도가 가장 센) `바르`·`크림`·`피부` 가 중심에 놓이고 그 사이의 **엣지가 가장 굵으며**, 중심 덩어리에서 떨어진 곳에 `자외선`—`차단` 이 자기들끼리 묶인 별도의 짝으로 나와 있는 것도 눈에 띈다. 즉 이 선크림의 리뷰는 **‘자외선 차단’이라는 기능** 축과 **‘끈적임 없이 부드럽게, 백탁 없이 발린다’는 사용감** 축, 두 갈래로 이야기된다.

**해설 — 읽을 때 주의할 점**

- bigram 상위에는 `쿠팡 체험`·`무료 제공`·`체험 이벤트` 같은 **리뷰 상투구**(체험단 고지 문구)도 섞여 있다. 빈도가 높다고 다 의미 있는 건 아니므로, 제품 속성과 무관한 상투구는 해석에서 빼거나 도메인 불용어 후보로 다시 검토한다.
- `바르 바르` 처럼 같은 어간이 이어진 묶음은 형태소 분석 결과가 잘게 쪼개져 생긴 것이라 의미가 없다.
- **흔한 실수 2가지**: 1) `binary=True` 를 빠뜨려 *등장 횟수의 곱* 을 세는 것(그러면 한 리뷰에서 여러 번 나온 단어가 과대평가된다), 2) 대각선을 0 으로 지우지 않아 자기 자신과의 카운트가 항상 1위로 올라오는 것.